<a href="https://colab.research.google.com/github/Sergi-e/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API key set up
from google.colab import userdata
API_KEY = userdata.get('GROQ_API_KEY')

# # TODO: set API_KEY using ONE of the methods above.
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

In [3]:
# Part 1.1 — My first API call

# TODO: helper function I'll reuse for the whole lab
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [4]:
# TODO: call it once with a simple question and print the answer
answer = ask_llm("What is a good name for a savings product for market traders in Accra?")
print(answer)

For a savings product targeting market traders in Accra, you'll want a name that resonates with the local culture and conveys a sense of security, trust, and growth. Here are some suggestions:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name would immediately connect with market traders.
2. **TradeSafe**: This name emphasizes the idea of keeping savings safe and secure, which is essential for market traders who often deal with cash.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect." This name suggests a product that helps traders gather and grow their savings.
4. **MarketMoen**: "Moen" is a Ghanaian word for "progress" or "growth." This name conveys the idea of helping market traders make progress with their savings.
5. **Adwuma Savings**: "Adwuma" means "work" or "business" in the Akan language. This name highlights the product's focus on supporting market traders' businesses.
6. **Kaya Savings**: "Kaya"

In [5]:
# TODO: print response.usage to see how many tokens the call consumed
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a good name for a savings product for market traders in Accra?"},
    ],
)
print(response.usage)

CompletionUsage(completion_tokens=398, prompt_tokens=57, total_tokens=455, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.134356285, prompt_time=0.002698601, completion_time=1.2850292589999999, total_time=1.28772786)


> **Student Reasoning — Anatomy of a call:**
>
> **1. System vs user roles:** The system role sets the model's identity and rules for the whole conversation, while the user role is the actual request being made in that turn. In my ask_llm() function, the system role tells the model to act as a helpful assistant, and the user role asks the specific question about a savings product name for market traders in Accra.
>
> **2. Tokens and billing:** A token is roughly a piece of a word, sometimes a whole short word and sometimes just a few characters, that the model reads and generates one at a time. My test call used 57 tokens for the prompt and 398 for the completion, for a total of 455. Providers bill per token instead of per request because the actual computing cost depends on how much text is processed and generated, not on the fact that a request was made. A one word answer and an eight paragraph list, like the one I got back, cost very different amounts to produce, so token based billing reflects that difference fairly.

In [6]:
# Part 1.2 — temperature: the randomness dial
# TODO: ask the same question 5 times at temperature=0.0 and 5 times at temperature=1.2
savings_question = "Suggest a name for a savings product for market traders in Accra."

low_temp_runs = [ask_llm(savings_question, temperature=0.0) for _ in range(5)]
high_temp_runs = [ask_llm(savings_question, temperature=1.2) for _ in range(5)]

In [7]:
# TODO: print all 10 answers, grouped by temperature
print("Temperature 0.0 runs")
for i, ans in enumerate(low_temp_runs, 1):
    print(f"Run {i}: {ans}")

print("\nTemperature 1.2 runs")
for i, ans in enumerate(high_temp_runs, 1):
    print(f"Run {i}: {ans}")

Temperature 0.0 runs
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the id

> **Student Reasoning — Temperature:**
>
> **1. What I observed:** At temperature 0.0, the answers were nearly identical across runs, the same names kept repeating, and two runs came back word for word the same. At temperature 1.2, the answers varied a lot more, new names appeared that never showed up at temperature 0, and even small details like word meanings shifted between runs.
>
> **2. Which temperature fits the loan decision-support system:** Temperature 0.0 is the right choice for that system. A loan officer needs consistent, repeatable output when reviewing an application, not a different summary or recommendation each time the same letter is processed. High temperature is useful for brainstorming, like generating product name ideas, but it works against reliability in a decision-support context, where trust depends on the system giving the same read on the same facts every time.

# Section 2 — The Dataset: Loan Application Letters

In [12]:
# TODO: load the six loan application letters and gold-standard labels
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


# Section 3 — Prompt Engineering for the Decision Support System

In [13]:
# Part 3.1 — Component 1: Summarization
# TODO: naive first attempt, run on L002 and L006
def summarize_v1(letter_text):
    return ask_llm(f"Summarize this: {letter_text}")

summary_l002_v1 = summarize_v1(LETTERS["L002"])
summary_l006_v1 = summarize_v1(LETTERS["L006"])

print("L002 V1:", summary_l002_v1)
print("\nL006 V1:", summary_l006_v1)

L002 V1: Kwame Boateng, a commercial driver in Kumasi, is seeking an urgent loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.

L006 V1: Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.


In [14]:
# TODO: proper V2 template with a role and constraints, run at temperature=0
summary_system_prompt = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan applications factually and neutrally, in 3 to 4 sentences. "
    "Do not invent or assume details that are not stated in the letter."
)

def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    return ask_llm(user_prompt, system_prompt=summary_system_prompt, temperature=0)

summary_l002_v2 = summarize_v2(LETTERS["L002"])
summary_l006_v2 = summarize_v2(LETTERS["L006"])

print("L002 V2:", summary_l002_v2)
print("\nL006 V2:", summary_l006_v2)

L002 V2: Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is seeking urgent assistance with the loan.

L006 V2: Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business. He is 22 years old and claims to be business-minded, citing feedback from his friends. Kofi has not yet started any of these businesses and plans to repay the loan in one year. He does not have collateral to offer, but asserts that he is trustworthy.


In [15]:
# TODO: compare V1 vs V2 side by side
print("L002 comparison")
print("V1:", summary_l002_v1)
print("V2:", summary_l002_v2)

print("\nL006 comparison")
print("V1:", summary_l006_v1)
print("V2:", summary_l006_v2)

L002 comparison
V1: Kwame Boateng, a commercial driver in Kumasi, is seeking an urgent loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.
V2: Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is seeking urgent assistance with the loan.

L006 comparison
V1: Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promise

> **Student Reasoning — Summarization prompts:**
>
> **1. Problems in V1 that V2 fixed:** V1 added small framing not actually in the letter. For L002, V1 wrote that Kwame "promises to repay the loan as soon as possible," but the letter says "I can pay back whenever the money comes," which is a much less certain statement. For L006, V1 wrote that Kofi promises to repay "once his businesses are successful," which is another confident phrase the letter never uses. V2 stayed closer to the literal wording in both cases.
>
> **2. Why "no invented details" is necessary, and the failure mode name:** A wording that sounds more certain or positive than the source could push a loan officer toward trusting an application more than what the letter actually supports. This is called hallucination in the LLM literature: when a model states something that is not grounded in the source text. Since the officer is making real financial decisions off this output, I think keeping it strictly factual is a good safeguard.